# California Wildfire Economic Forecasting Project
This notebook walks through the full machine learning pipeline:
- Load & clean raw wildfire, consumer spending (PCE), and housing price index (HPI) data
- Merge into a unified dataset
- Train models to predict HPI and PCE from wildfire severity
- Run custom simulations using a forecast function


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

import numpy as np
from IPython.display import display, Markdown
from ipywidgets import interact, IntSlider, Dropdown

In [ ]:
wildfire_df = pd.read_csv('data/California Wildfire Damage.csv')
pce_raw_df = pd.read_csv('data/Table.csv', header=None, skiprows=4)
hpi_df = pd.read_csv('data/hpi_master.csv')

In [ ]:
wildfire_df['Date'] = pd.to_datetime(wildfire_df['Date'], errors='coerce')
wildfire_df['Year'] = wildfire_df['Date'].dt.year
wildfire_agg = wildfire_df.groupby('Year').agg({
    'Area_Burned (Acres)': 'sum',
    'Homes_Destroyed': 'sum',
    'Fatalities': 'sum',
    'Estimated_Financial_Loss (Million $)': 'sum'
}).reset_index()

## Process Consumer Spending (PCE) Data

In [ ]:
pce_raw_df.columns = ['GeoFips', 'GeoName', 'LineCode', 'Description'] + list(range(1997, 2024))
pce_ca = pce_raw_df[pce_raw_df['GeoFips'] == '06000']
pce_melted = pce_ca.melt(id_vars=['LineCode', 'Description'], var_name='Year', value_name='Spending')
pce_melted = pce_melted[pce_melted['Year'].apply(lambda x: str(x).isdigit())]
pce_melted['Year'] = pce_melted['Year'].astype(int)
pce_pivot = pce_melted.pivot(index='Year', columns='Description', values='Spending').reset_index()

## Process Housing Price Index (HPI) Data

In [ ]:
hpi_df = hpi_df[hpi_df['frequency'] == 'monthly']
hpi_df['Month'] = pd.to_datetime(hpi_df['yr'].astype(str) + '-' + hpi_df['period'].astype(str) + '-01', errors='coerce')
hpi_df['Year'] = hpi_df['Month'].dt.year
hpi_agg = hpi_df.groupby('Year')['index_nsa'].mean().reset_index().rename(columns={'index_nsa': 'Avg_HPI'})

## Merge Final Dataset

In [ ]:
final_df = wildfire_agg.merge(pce_pivot, on='Year', how='inner').merge(hpi_agg, on='Year', how='inner')
final_df.to_csv('cleaned_merged_dataset.csv', index=False)
final_df.head()

In [ ]:
category_columns = [
    'Durable goods',
    'Pharmaceutical and other medical products',
    'Construction'
]

category_columns = [col for col in category_columns if col in pce_pivot.columns]
final_df = final_df.merge(pce_pivot[['Year'] + category_columns], on='Year', how='left')


## Train ML Models

In [ ]:
features = ['Area_Burned (Acres)', 'Homes_Destroyed', 'Fatalities', 'Estimated_Financial_Loss (Million $)']
X = final_df[features]
y_hpi = final_df['Avg_HPI']
y_pce = final_df['Personal consumption expenditures']

model_hpi = GradientBoostingRegressor(random_state=42).fit(X, y_hpi)
model_pce = GradientBoostingRegressor(random_state=42).fit(X, y_pce)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)

## Forecast Function

In [ ]:
def forecast_wildfire_impact(area_burned, homes_destroyed, fatalities, financial_loss):
    input_df = pd.DataFrame([{
        'Area_Burned (Acres)': area_burned,
        'Homes_Destroyed': homes_destroyed,
        'Fatalities': fatalities,
        'Estimated_Financial_Loss (Million $)': financial_loss
    }])
    pred_hpi = model_hpi.predict(input_df)[0]
    pred_pce = model_pce.predict(input_df)[0]
    cluster = kmeans.predict(scaler.transform(input_df))[0]
    return {
        'Predicted HPI': round(pred_hpi, 2),
        'Predicted PCE ($B)': round(pred_pce, 0),
        'Cluster Type': int(cluster)
    }

## Example Wildfire Simulation

In [ ]:
result = forecast_wildfire_impact(600000, 11000, 90, 35000)
for k, v in result.items():
    print(f'{k}: {v}')

In [ ]:
final_df = pd.read_csv('cleaned_merged_dataset.csv')

# Define input features and targets
features = ['Area_Burned (Acres)', 'Homes_Destroyed', 'Fatalities', 'Estimated_Financial_Loss (Million $)']
X = final_df[features]
y_hpi = final_df['Avg_HPI']
y_pce = final_df['Personal consumption expenditures']

# HPI Prediction Model
model_hpi = GradientBoostingRegressor(random_state=42)
model_hpi.fit(X, y_hpi)
final_df['HPI_Predicted'] = model_hpi.predict(X)
final_df['HPI_Error'] = final_df['Avg_HPI'] - final_df['HPI_Predicted']
hpi_r2 = r2_score(y_hpi, final_df['HPI_Predicted'])

# PCE Prediction Model
model_pce = GradientBoostingRegressor(random_state=42)
model_pce.fit(X, y_pce)
final_df['PCE_Predicted'] = model_pce.predict(X)
final_df['PCE_Error'] = final_df['Personal consumption expenditures'] - final_df['PCE_Predicted']
pce_r2 = r2_score(y_pce, final_df['PCE_Predicted'])

# KMeans Clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
final_df['Cluster'] = kmeans.fit_predict(X_scaled)

# PCA for Visualization
pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame(pca_components, columns=['PC1', 'PC2'])
pca_df['Year'] = final_df['Year']
pca_df['Cluster'] = final_df['Cluster']

# Summary Table Output
summary_df = pd.DataFrame({
    "Task": [
        "HPI Prediction",
        "PCE Prediction",
        "Grouping Similar Years",
        "Exploratory Patterns"
    ],
    "Best Model": [
        "Gradient Boosting Regressor",
        "Gradient Boosting Regressor",
        "KMeans Clustering (k=3)",
        "PCA"
    ],
    "Why": [
        f"R² = {hpi_r2:.2f}, handles nonlinear wildfire–housing trends",
        f"R² = {pce_r2:.2f}, captures post-fire spending shifts",
        "Identifies economic response archetypes per year",
        f"Explains {pca.explained_variance_ratio_.sum():.2f} of variance across 2 dimensions"
    ]
})

#Display everything
print("Full Analysis Complete\n")
display(summary_df)
display(final_df.head())


In [ ]:
# Load raw datasets
wildfire_df = pd.read_csv('data/California Wildfire Damage.csv')
pce_raw_df = pd.read_csv('data/Table.csv', header=None, skiprows=4)
hpi_df = pd.read_csv('data/hpi_master.csv')

# Clean Wildfire Data
wildfire_df['Date'] = pd.to_datetime(wildfire_df['Date'], errors='coerce')
wildfire_df['Year'] = wildfire_df['Date'].dt.year
wildfire_agg = wildfire_df.groupby('Year').agg({
    'Area_Burned (Acres)': 'sum',
    'Homes_Destroyed': 'sum',
    'Fatalities': 'sum',
    'Estimated_Financial_Loss (Million $)': 'sum'
}).reset_index()

# Clean PCE Data
pce_raw_df.columns = ['GeoFips', 'GeoName', 'LineCode', 'Description'] + list(range(1997, 2024))
pce_ca = pce_raw_df[pce_raw_df['GeoFips'] == '06000']
pce_melted = pce_ca.melt(id_vars=['LineCode', 'Description'], var_name='Year', value_name='Spending')
pce_melted = pce_melted[pce_melted['Year'].apply(lambda x: str(x).isdigit())]
pce_melted['Year'] = pce_melted['Year'].astype(int)
pce_pivot = pce_melted.pivot(index='Year', columns='Description', values='Spending').reset_index()

# Clean HPI Data
hpi_df = hpi_df[hpi_df['frequency'] == 'monthly']
hpi_df['Month'] = pd.to_datetime(hpi_df['yr'].astype(str) + '-' + hpi_df['period'].astype(str) + '-01', errors='coerce')
hpi_df['Year'] = hpi_df['Month'].dt.year
hpi_agg = hpi_df.groupby('Year')['index_nsa'].mean().reset_index().rename(columns={'index_nsa': 'Avg_HPI'})

# Merge Final Dataset
final_df = wildfire_agg.merge(pce_pivot, on='Year', how='inner').merge(hpi_agg, on='Year', how='inner')

# Train Models
features = ['Area_Burned (Acres)', 'Homes_Destroyed', 'Fatalities', 'Estimated_Financial_Loss (Million $)']
X = final_df[features]
y_hpi = final_df['Avg_HPI']
y_pce = final_df['Personal consumption expenditures']

model_hpi = GradientBoostingRegressor(random_state=42).fit(X, y_hpi)
model_pce = GradientBoostingRegressor(random_state=42).fit(X, y_pce)

final_df['HPI_Predicted'] = model_hpi.predict(X)
final_df['HPI_Error'] = final_df['Avg_HPI'] - final_df['HPI_Predicted']
final_df['PCE_Predicted'] = model_pce.predict(X)
final_df['PCE_Error'] = final_df['Personal consumption expenditures'] - final_df['PCE_Predicted']

# KMeans Clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
final_df['Cluster'] = kmeans.fit_predict(X_scaled)

# PCA for Visualization
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'])
pca_df['Year'] = final_df['Year']
pca_df['Cluster'] = final_df['Cluster']

# Feature Importance Visualization
plt.figure(figsize=(10, 4))
sns.barplot(x=model_hpi.feature_importances_, y=features)
plt.title("Feature Importance for HPI Prediction")
plt.xlabel("Importance Score")
plt.ylabel("Wildfire Feature")
plt.show()

#nCategory-level Post-Wildfire Spending by Cluster
# Pick best-match columns from pce_pivot:
pce_columns = [col for col in final_df.columns if "durable" in col.lower() or "pharma" in col.lower() or "construct" in col.lower()]
if pce_columns:
    cluster_means = final_df.groupby('Cluster')[pce_columns].mean().reset_index()
    cluster_melted = cluster_means.melt(id_vars='Cluster', var_name='Category', value_name='Avg Spending')

    plt.figure(figsize=(10, 6))
    sns.barplot(data=cluster_melted, x='Category', y='Avg Spending', hue='Cluster')
    plt.title("Category spending by cluster")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("No matching columns found for durable goods, pharmaceuticals, or construction.")

# Key Insight Text Summary
display(Markdown("""
## Key Insights

- **Fatalities** and **Financial Loss** are the top predictors of both HPI and PCE.
- Cluster **0** = *Strong recovery years* (high housing & spending rebound).
- Post-wildfire surges most visible in:
  - Durable Goods (furniture, vehicles)
  - Health & Pharmaceuticals
  - Construction Services
"""))

In [ ]:
model_hpi = GradientBoostingRegressor(random_state=42).fit(X, y_hpi)
model_pce = GradientBoostingRegressor(random_state=42).fit(X, y_pce)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)

In [ ]:
features = ['Area_Burned (Acres)', 'Homes_Destroyed', 'Fatalities', 'Estimated_Financial_Loss (Million $)']
X = final_df[features]
y_hpi = final_df['Avg_HPI']
y_pce = final_df['Personal consumption expenditures']

model_hpi = GradientBoostingRegressor(random_state=42).fit(X, y_hpi)
model_pce = GradientBoostingRegressor(random_state=42).fit(X, y_pce)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)

In [ ]:
# Load and Clean datasets
wildfire_df = pd.read_csv('data/California Wildfire Damage.csv')
pce_raw_df = pd.read_csv('data/Table.csv', header=None, skiprows=4)
hpi_df = pd.read_csv('data/hpi_master.csv')

wildfire_df['Date'] = pd.to_datetime(wildfire_df['Date'], errors='coerce')
wildfire_df['Year'] = wildfire_df['Date'].dt.year
wildfire_agg = wildfire_df.groupby('Year').agg({
    'Area_Burned (Acres)': 'sum',
    'Homes_Destroyed': 'sum',
    'Fatalities': 'sum',
    'Estimated_Financial_Loss (Million $)': 'sum'
}).reset_index()

pce_raw_df.columns = ['GeoFips', 'GeoName', 'LineCode', 'Description'] + list(range(1997, 2024))
pce_ca = pce_raw_df[pce_raw_df['GeoFips'] == '06000']
pce_melted = pce_ca.melt(id_vars=['LineCode', 'Description'], var_name='Year', value_name='Spending')
pce_melted['Year'] = pd.to_numeric(pce_melted['Year'], errors='coerce')
pce_pivot = pce_melted.pivot_table(
    index='Year',
    columns='Description',
    values='Spending',
    aggfunc='sum'  # or 'mean' if you prefer
).reset_index()


hpi_df = hpi_df[hpi_df['frequency'] == 'monthly']
hpi_df['Month'] = pd.to_datetime(hpi_df['yr'].astype(str) + '-' + hpi_df['period'].astype(str) + '-01', errors='coerce')
hpi_df['Year'] = hpi_df['Month'].dt.year
hpi_agg = hpi_df.groupby('Year')['index_nsa'].mean().reset_index().rename(columns={'index_nsa': 'Avg_HPI'})

final_df = wildfire_agg.merge(pce_pivot, on='Year', how='inner').merge(hpi_agg, on='Year', how='inner')

# Model Training
features = ['Area_Burned (Acres)', 'Homes_Destroyed', 'Fatalities', 'Estimated_Financial_Loss (Million $)']
X = final_df[features]
y_hpi = final_df['Avg_HPI']
y_pce = final_df['Personal consumption expenditures']

model_hpi = GradientBoostingRegressor(random_state=42).fit(X, y_hpi)
model_pce = GradientBoostingRegressor(random_state=42).fit(X, y_pce)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)
final_df['Cluster'] = kmeans.predict(X_scaled)

# Forecast + AI Insight Generator
def forecast_wildfire_impact_verbose(area_burned, homes_destroyed, fatalities, financial_loss):
    input_df = pd.DataFrame([{
        'Area_Burned (Acres)': area_burned,
        'Homes_Destroyed': homes_destroyed,
        'Fatalities': fatalities,
        'Estimated_Financial_Loss (Million $)': financial_loss
    }])
    pred_hpi = model_hpi.predict(input_df)[0]
    pred_pce = model_pce.predict(input_df)[0]
    cluster = kmeans.predict(scaler.transform(input_df))[0]

    insights = []
    for col, val in zip(features, [area_burned, homes_destroyed, fatalities, financial_loss]):
        percentile = (final_df[col] < val).mean() * 100
        if percentile > 90:
            insights.append(f"{col} is extremely high (above {int(percentile)}% of historical wildfires).")
        elif percentile > 70:
            insights.append(f"{col} is above average (above {int(percentile)}%).")
        elif percentile < 30:
            insights.append(f"{col} is relatively low (below {int(100 - percentile)}%).")

    cluster_descriptions = {
        0: "Cluster 0: Strong post-fire rebound (durables, construction, health).",
        1: "Cluster 1: Moderate recovery, selective sector growth.",
        2: "Cluster 2: Weak response, risk of economic stagnation."
    }
    insights.append(cluster_descriptions[cluster])

    if cluster == 0:
        summary = "Outlook: Expect strong spending in construction, vehicles, and health. Real estate likely to rise."
    elif cluster == 1:
        summary = "Outlook: Moderate growth. Some sectors rebound. Real estate stable or regional."
    else:
        summary = "Outlook: Low confidence. Spending constrained. Real estate may dip or remain flat."

    insights.append(summary)
    return round(pred_hpi, 2), round(pred_pce, 0), int(cluster), insights

# Interactive Simulation Widget
preset_profiles = {
    "Mild Fire": {"acres": 100000, "homes": 500, "fatalities": 5, "loss": 3000},
    "Moderate Fire": {"acres": 300000, "homes": 5000, "fatalities": 25, "loss": 15000},
    "Severe Fire": {"acres": 600000, "homes": 12000, "fatalities": 85, "loss": 32000},
    "Extreme Fire": {"acres": 900000, "homes": 20000, "fatalities": 150, "loss": 50000},
    "Custom": None
}

def interactive_forecast_with_insight(preset, acres, homes, fatalities, loss):
    if preset != "🛠 Custom":
        profile = preset_profiles[preset]
        acres, homes, fatalities, loss = profile["acres"], profile["homes"], profile["fatalities"], profile["loss"]

    hpi, pce, cluster, insights = forecast_wildfire_impact_verbose(acres, homes, fatalities, loss)

    print("Forecast Results")
    print(f"Predicted HPI: {hpi}")
    print(f"Predicted PCE: ${pce:,}")
    print(f"Recovery Cluster: {cluster}")

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(final_df['Avg_HPI'], kde=True, color='skyblue', ax=ax[0])
    ax[0].axvline(hpi, color='red', linestyle='--', label='Predicted HPI')
    ax[0].set_title("Predicted vs Historical HPI")
    ax[0].legend()

    sns.histplot(final_df['Personal consumption expenditures'], kde=True, color='lightgreen', ax=ax[1])
    ax[1].axvline(pce, color='darkgreen', linestyle='--', label='Predicted PCE')
    ax[1].set_title("Predicted vs Historical PCE")
    ax[1].legend()
    plt.tight_layout()
    plt.show()

    print("\nAI generated insights:")
    for line in insights:
        print("- " + line)

interact(
    interactive_forecast_with_insight,
    preset=Dropdown(options=list(preset_profiles.keys()), value="Moderate Fire", description="Preset"),
    acres=IntSlider(min=50000, max=1000000, step=50000, value=300000, description='Acres'),
    homes=IntSlider(min=0, max=20000, step=500, value=5000, description='Homes'),
    fatalities=IntSlider(min=0, max=200, step=5, value=25, description='Fatalities'),
    loss=IntSlider(min=1000, max=60000, step=1000, value=15000, description='Loss ($M)')
)

In [ ]:
blackrock_palette = ['#000000', '#8C52FF', '#FFA500', '#FFD700']
sns.set_style("whitegrid")
plt.rcParams.update({'font.family': 'Arial', 'axes.titlesize': 14, 'axes.labelsize': 12})

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=pca_df['PC1'], y=pca_df['PC2'],
    hue=pca_df['Cluster'],
    palette=blackrock_palette[:3],
    s=100, edgecolor='gray'
)
plt.title("PCA of Wildfire Impact Data by Cluster")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title='Cluster')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.countplot(
    x='Cluster',
    data=final_df,
    palette=blackrock_palette[:3]
)
plt.title("Number of Years per Cluster (KMeans)")
plt.xlabel("Cluster Label")
plt.ylabel("Number of Years")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(
    x=model_hpi.feature_importances_,
    y=features,
    palette=blackrock_palette[::-1]
)
plt.title("Feature Importance for Predicting HPI")
plt.xlabel("Importance Score")
plt.ylabel("Wildfire Feature")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(
    x=model_pce.feature_importances_,
    y=features,
    palette=blackrock_palette[::-1]
)
plt.title("Feature Importance for Predicting PCE")
plt.xlabel("Importance Score")
plt.ylabel("Wildfire Feature")
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown("""
## PCA (Principal Component Analysis)

This scatterplot shows the first two principal components derived from the wildfire data.

- **Purpose**: Reduce the dimensionality of the data to visualize how years cluster based on wildfire impact.
- **Insight**: Distinct clusters appear in PC1-PC2 space, confirming that wildfire metrics create meaningful economic groupings.
- Years in the same cluster show similar fire intensity and recovery outcomes.
"""))

display(Markdown("""
## KMeans Clustering (Year Groups)

This bar chart shows how many years fall into each of the 3 recovery clusters.

- **Purpose**: Group years based on severity and impact of wildfires.
- **Insight**:
  - Cluster 0 → Strong recovery years (surge in HPI & PCE).
  - Cluster 1 → Moderate/uneven recovery.
  - Cluster 2 → Weak or stalled recovery.
- Cluster analysis helps inform investment, aid, or infrastructure response planning.
"""))

display(Markdown("""
## Gradient Boosting: Feature Importance for HPI

This chart shows which wildfire features had the greatest influence on predicting housing prices.

- **Top Predictors**: Fatalities and Financial Loss.
- **Insight**: Housing prices tend to increase when fire-related losses and fatality counts are high — likely due to destruction of supply.
- Gradient Boosting captured these nonlinear relationships better than linear models.
"""))

display(Markdown("""
## Gradient Boosting: Feature Importance for PCE

This chart shows which wildfire features drove changes in consumer spending (PCE).

- **Top Predictors**: Financial Loss and Area Burned.
- **Insight**: Post-fire periods show spikes in consumption — especially in construction, healthcare, and durable goods.
- This helps forecast budget and infrastructure needs following severe fire seasons.
"""))